In [ ]:
# Cell 1: Setup, create zero labels for controls only, and collect sample metadata (strict patient label requirement)
import os
from pathlib import Path
import numpy as np
import cv2
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader

# Base data directories
DATA_ROOT = Path("data")
CONTROLS_IMG_DIR = DATA_ROOT / "controls" / "imgs"
CONTROLS_LABEL_DIR = DATA_ROOT / "controls" / "labels"  # will hold the synthetic zero masks
PATIENTS_IMG_DIR = DATA_ROOT / "patients" / "imgs"
PATIENTS_LABEL_DIR = DATA_ROOT / "patients" / "labels"

# Ensure control label directory exists
os.makedirs(CONTROLS_LABEL_DIR, exist_ok=True)

# Helper to read an image as grayscale robustly
def read_gray_image(path):
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        tmp = cv2.imread(str(path), cv2.IMREAD_COLOR)
        if tmp is None:
            raise RuntimeError(f"Failed to read image {path}")
        img = cv2.cvtColor(tmp, cv2.COLOR_BGR2GRAY)
    return img

# Create zero (healthy) labels for controls if missing
def ensure_control_labels():
    control_imgs = sorted(list(CONTROLS_IMG_DIR.glob("*")))
    for img_path in control_imgs:
        img = read_gray_image(img_path)
        zero_label = np.zeros_like(img, dtype=np.uint8)
        label_path = CONTROLS_LABEL_DIR / img_path.name
        if not label_path.exists():
            cv2.imwrite(str(label_path), zero_label)

# Run creation for control labels
ensure_control_labels()

# Build sample list
samples = []

# Patients: require existing labels; collect all missing and fail if any
patient_imgs = sorted(list(PATIENTS_IMG_DIR.glob("*")))
missing_patient_labels = []
for img_path in patient_imgs:
    label_path = PATIENTS_LABEL_DIR / img_path.name
    if not label_path.exists():
        missing_patient_labels.append(img_path.name)
    else:
        samples.append({
            "img": img_path,
            "label": label_path,
            "is_control": False
        })

if missing_patient_labels:
    raise FileNotFoundError(
        f"Missing label(s) for patient image(s): {missing_patient_labels}. "
        "All patient images must have corresponding ground-truth labels."
    )

# Controls: include with their zero labels
control_imgs = sorted(list(CONTROLS_IMG_DIR.glob("*")))
for img_path in control_imgs:
    label_path = CONTROLS_LABEL_DIR / img_path.name
    if not label_path.exists():
        raise RuntimeError(f"Control label unexpectedly missing for {img_path.name}")
    samples.append({
        "img": img_path,
        "label": label_path,
        "is_control": True
    })

print(f"Total samples collected: {len(samples)} (patients: {len(patient_imgs)}, controls: {len(control_imgs)})")

In [ ]:
# Cell 2: Dataset class + split + dataloaders with robust resizing and safe collate

import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import cv2
import numpy as np

# Target spatial size for all images/labels (H, W)
TARGET_SIZE = (256, 256)  # adjust to your network's expected input

def resize_image(img: np.ndarray, size):
    """
    Resize image to size (H, W). Accepts:
      - CHW (e.g., (1,H,W) or (3,H,W))
      - HWC (e.g., (H,W,1) or (H,W,3))
      - HW (2D)
    Returns CHW.
    """
    if img.ndim == 3:
        # Determine if it's CHW or HWC: if first dim is small (<=4) treat as CHW
        if img.shape[0] in (1, 3, 4):
            # CHW -> HWC for cv2
            img_hwc = img.transpose(1, 2, 0)
        else:
            # assume HWC
            img_hwc = img
        resized_hwc = cv2.resize(img_hwc, (size[1], size[0]), interpolation=cv2.INTER_LINEAR)
        
        # Handle the case where cv2.resize returns 2D array for single channel
        if resized_hwc.ndim == 2:
            resized_chw = np.expand_dims(resized_hwc, 0)  # Add channel dimension
        else:
            # convert back to CHW
            resized_chw = resized_hwc.transpose(2, 0, 1)
    elif img.ndim == 2:
        # single channel
        resized2 = cv2.resize(img, (size[1], size[0]), interpolation=cv2.INTER_LINEAR)
        resized_chw = np.expand_dims(resized2, 0)
    else:
        raise ValueError(f"Unsupported image ndim {img.ndim}")
    return resized_chw

def resize_label(lbl: np.ndarray, size):
    """
    Resize label to size (H,W). Expect input like (1,H,W) or (H,W), returns (1,H,W) binary.
    """
    if lbl.ndim == 3 and lbl.shape[0] in (1,):
        arr = lbl.squeeze(0)
    elif lbl.ndim == 2:
        arr = lbl
    else:
        raise ValueError(f"Unsupported label shape {lbl.shape}")
    resized = cv2.resize(arr.astype(np.uint8), (size[1], size[0]), interpolation=cv2.INTER_NEAREST)
    binarized = (resized > 0).astype(np.uint8)
    return np.expand_dims(binarized, 0)

class TumorSegmentationDataset(Dataset):
    def __init__(self, samples, resize_to=None):
        """
        samples: list of dicts with keys 'img', 'label', 'is_control'
        resize_to: tuple (H, W) to force all samples to fixed size
        """
        self.samples = samples
        self.resize_to = resize_to

    def __len__(self):
        return len(self.samples)

    def _load_image(self, path):
        img = cv2.imread(str(path), cv2.IMREAD_ANYDEPTH)
        if img is None:
            img = cv2.imread(str(path), cv2.IMREAD_COLOR)
            if img is None:
                raise RuntimeError(f"Failed to read image {path}")
            img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        img = img.astype(np.float32)
        maxv = np.max(img)
        if maxv > 0:
            img /= maxv
        img = np.expand_dims(img, axis=0)  # CHW
        if self.resize_to is not None:
            img = resize_image(img, self.resize_to)
        return img

    def _load_label(self, path):
        lbl = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if lbl is None:
            raise RuntimeError(f"Failed to read label {path}")
        lbl = (lbl > 0).astype(np.uint8)
        lbl = np.expand_dims(lbl, axis=0)  # (1,H,W)
        if self.resize_to is not None:
            lbl = resize_label(lbl, self.resize_to)
        return lbl

    def __getitem__(self, idx):
        sample = self.samples[idx]
        img = self._load_image(sample["img"])        # (1,H,W)
        label = self._load_label(sample["label"])    # (1,H,W)
        img_tensor = torch.from_numpy(img)           # float32
        label_tensor = torch.from_numpy(label).float()
        return img_tensor, label_tensor, sample["is_control"]

def collate_fn(batch):
    imgs, labels, is_controls = zip(*batch)
    imgs = torch.stack(imgs, dim=0)       # (B,1,H,W)
    labels = torch.stack(labels, dim=0)   # (B,1,H,W)
    is_controls = torch.tensor([int(x) for x in is_controls], dtype=torch.bool)
    return imgs, labels, is_controls

# Stratified split to preserve control/patient ratio
is_control_flags = [int(s["is_control"]) for s in samples]
train_samples, val_samples = train_test_split(
    samples,
    test_size=0.2,
    stratify=is_control_flags,
    random_state=42
)

train_dataset = TumorSegmentationDataset(train_samples, resize_to=TARGET_SIZE)
val_dataset = TumorSegmentationDataset(val_samples, resize_to=TARGET_SIZE)

# DataLoader: num_workers=0 avoids multiprocessing pickling issues in notebook; pin_memory only if CUDA available
use_pin_memory = torch.cuda.is_available()
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0,
    pin_memory=use_pin_memory,
    collate_fn=collate_fn
)
val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    pin_memory=use_pin_memory,
    collate_fn=collate_fn
)

print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}")

# Peek at one batch
batch = next(iter(train_loader))
imgs, labels, is_controls = batch
print("Batch shapes:", imgs.shape, labels.shape)  # (B,1,H,W)
print("Control flags in batch:", is_controls)

In [ ]:
# Cell 3: Build nnU-Net png dataset + dataset.json for 5-fold CV (controls have zero label, patients keep their ground truth)
import json
from pathlib import Path
import cv2
import numpy as np
import shutil

# Parameters: adjust if you want a different task ID / name
DATASET_ID = 123  # pick unused three-digit ID
DATASET_NAME = "MyTumor"
NNUNET_RAW_BASE = Path("nnUNet_raw")  # you can also set via env vars in shell later
TASK_FOLDER = NNUNET_RAW_BASE / f"Dataset{DATASET_ID:03d}_{DATASET_NAME}"

# Paths
imagesTr_dir = TASK_FOLDER / "imagesTr"
labelsTr_dir = TASK_FOLDER / "labelsTr"
imagesTs_dir = TASK_FOLDER / "imagesTs"  # optional; we leave it empty

# Create folders
for p in (imagesTr_dir, labelsTr_dir):
    p.mkdir(parents=True, exist_ok=True)

# Build mapping: case identifier -> sample
training_entries = []
for idx, sample in enumerate(samples):
    case_id = f"case_{idx:04d}"
    # Image: single channel -> modality code 0000
    src_img = sample["img"]
    dst_img_name = f"{case_id}_0000.png"
    dst_img_path = imagesTr_dir / dst_img_name
    shutil.copyfile(src_img, dst_img_path)

    # Label: read, binarize to 0/1, save as PNG (nnU-Net expects integer map)
    src_label = sample["label"]
    lbl = cv2.imread(str(src_label), cv2.IMREAD_GRAYSCALE)
    if lbl is None:
        raise RuntimeError(f"Failed to load label {src_label}")
    bin_lbl = (lbl > 0).astype(np.uint8)  # 0 or 1
    # Save with values 0 and 1 (not 0/255)
    dst_label_name = f"{case_id}.png"
    dst_label_path = labelsTr_dir / dst_label_name
    # Multiplying by 1 keeps it 0/1; PNG will store it fine
    cv2.imwrite(str(dst_label_path), bin_lbl)

    # Append to dataset.json training list (relative paths)
    training_entries.append({
        "image": f"./imagesTr/{dst_img_name}",
        "label": f"./labelsTr/{dst_label_name}"
    })

# Build dataset.json
dataset_json = {
    "channel_names": {
        "0": "PET"  # or whatever modality makes sense; influences normalization
    },
    "labels": {
        "background": 0,
        "tumor": 1
    },
    "numTraining": len(training_entries),
    "training": training_entries,
    "test": [],
    "file_ending": ".png"
    # optionally: "overwrite_image_reader_writer": "NaturalImage2DIO"
}

# Save dataset.json
with open(TASK_FOLDER / "dataset.json", "w") as f:
    json.dump(dataset_json, f, indent=2)

print(f"Created nnU-Net dataset at {TASK_FOLDER}")
print(f"Total training cases: {len(training_entries)} (controls and patients combined)")

In [ ]:
# Cell 3: Compare original vs preprocessed image

import matplotlib.pyplot as plt
import numpy as np
import torch
import cv2

# Assumes TARGET_SIZE was defined in Cell 2, e.g., (H, W)
# If not, define a fallback:
try:
    TARGET_SIZE
except NameError:
    TARGET_SIZE = (256, 256)

def hu_window(ct_img: np.ndarray, window_min: float, window_max: float) -> np.ndarray:
    """
    Apply HU windowing: clip to [window_min, window_max] then scale to [0,1].
    Assumes ct_img is in HxW (raw HU values).
    """
    clipped = np.clip(ct_img, window_min, window_max)
    scaled = (clipped - window_min) / (window_max - window_min)
    return scaled.astype(np.float32)

def preprocess_image(raw_img: np.ndarray,
                     window: tuple[int, int] = (-1200, 600),
                     target_size: tuple[int, int] = TARGET_SIZE) -> np.ndarray:
    """
    1. Assume raw_img is single-channel (HU) image.
    2. Apply HU windowing.
    3. Resize to target_size (H,W).
    4. Keep shape as (1,H,W) for model input.
    """
    # Apply windowing
    windowed = hu_window(raw_img.astype(np.float32), window[0], window[1])  # HxW in [0,1]
    # Resize (preserve as single channel)
    resized = cv2.resize(windowed, (target_size[1], target_size[0]), interpolation=cv2.INTER_LINEAR)
    # Expand channel dim
    return np.expand_dims(resized, axis=0)  # (1,H,W)

def unnormalized_display(img_tensor: torch.Tensor) -> np.ndarray:
    """
    Convert model-style tensor (1,H,W) or (B,1,H,W) to displayable HxW image in [0,1].
    """
    if isinstance(img_tensor, torch.Tensor):
        img = img_tensor.detach().cpu().numpy()
    else:
        img = img_tensor
    if img.ndim == 4:  # batch
        img = img[0]
    if img.shape[0] == 1:
        img = img[0]
    # Clip to [0,1]
    img = np.clip(img, 0, 1)
    return img

# Pick one example from train_loader
batch = next(iter(train_loader))
imgs, labels, is_controls = batch  # imgs: (B,1,H,W)
# Convert back to numpy for "original" raw version.
# We need the raw (pre-resize/window) version: load it directly from disk for clarity.
sample_idx = 0
img_tensor, label_tensor, is_control_flag = imgs[sample_idx], labels[sample_idx], is_controls[sample_idx]
sample_info = train_dataset.samples[sample_idx]
raw_path = sample_info["img"]
raw_img = cv2.imread(str(raw_path), cv2.IMREAD_ANYDEPTH)
if raw_img is None:
    raw_img = cv2.imread(str(raw_path), cv2.IMREAD_COLOR)
    raw_img = cv2.cvtColor(raw_img, cv2.COLOR_BGR2GRAY)
# If the raw image is not in HU units (e.g., already normalized), this will still show differences.

# Preprocess the raw image with a representative window (e.g., lung window)
preprocessed = preprocess_image(raw_img, window=(-1200, 600), target_size=TARGET_SIZE)  # (1,H,W)

# For comparison: also create a "naively normalized" version like the dataset loader would (global scaling)
def naive_normalize(img: np.ndarray, target_size):
    img_f = img.astype(np.float32)
    maxv = np.max(img_f)
    if maxv > 0:
        img_f /= maxv
    resized = cv2.resize(img_f, (target_size[1], target_size[0]), interpolation=cv2.INTER_LINEAR)
    return np.expand_dims(resized, axis=0)

naive = naive_normalize(raw_img, TARGET_SIZE)  # (1,H,W)

# Prepare display
orig_display = cv2.resize(raw_img.astype(np.float32), (TARGET_SIZE[1], TARGET_SIZE[0]), interpolation=cv2.INTER_LINEAR)
# Normalize original for consistent display (min-max)
minv, maxv = orig_display.min(), orig_display.max()
if maxv > minv:
    orig_display_norm = (orig_display - minv) / (maxv - minv)
else:
    orig_display_norm = orig_display * 0.0

preprocessed_disp = preprocessed[0]  # (H,W)
naive_disp = naive[0]  # (H,W)

# Difference between naive and windowed
diff = preprocessed_disp - naive_disp  # can be negative

# Plot comparison
plt.figure(figsize=(12, 3))
plt.subplot(1,5,1)
plt.imshow(orig_display_norm, cmap="gray", vmin=0, vmax=1)
plt.title("Raw (rescaled)")
plt.axis("off")

plt.subplot(1,5,2)
plt.imshow(naive_disp, cmap="gray", vmin=0, vmax=1)
plt.title("Naive norm + resize")
plt.axis("off")

plt.subplot(1,5,3)
plt.imshow(preprocessed_disp, cmap="gray", vmin=0, vmax=1)
plt.title("HU windowed (lung)")
plt.axis("off")

plt.subplot(1,5,4)
# Show difference, centered at zero with diverging colormap
plt.imshow(diff, cmap="bwr", vmin=-0.5, vmax=0.5)
plt.title("Windowed - Naive")
plt.colorbar(fraction=0.046, pad=0.04)
plt.axis("off")

plt.subplot(1,5,5)
# Overlay: windowed as intensity, label contour if exists
plt.imshow(preprocessed_disp, cmap="gray", vmin=0, vmax=1)
# convert label to contour
label_np = label_tensor[0,0].detach().cpu().numpy()  # (H,W) after resize already in dataset
contours, _ = cv2.findContours((label_np > 0).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
for cnt in contours:
    cnt = cnt.squeeze(1)
    plt.plot(cnt[:,1], cnt[:,0], color="red", linewidth=1)
plt.title("Windowed + Label")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Cell 4: 5-fold cross-validation training loop with Dice metric (nnU-Net style)

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import Subset, DataLoader
import numpy as np
from tqdm import tqdm

# Simple U-Net backbone (2D) – can be swapped for nnU-Net implementation if available
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class UNetSimple(nn.Module):
    def __init__(self, in_channels=1, base_features=32):
        super().__init__()
        self.down1 = DoubleConv(in_channels, base_features)
        self.pool1 = nn.MaxPool2d(2)
        self.down2 = DoubleConv(base_features, base_features * 2)
        self.pool2 = nn.MaxPool2d(2)
        self.bridge = DoubleConv(base_features * 2, base_features * 4)
        self.up2 = nn.ConvTranspose2d(base_features * 4, base_features * 2, 2, stride=2)
        self.dec2 = DoubleConv(base_features * 4, base_features * 2)
        self.up1 = nn.ConvTranspose2d(base_features * 2, base_features, 2, stride=2)
        self.dec1 = DoubleConv(base_features * 2, base_features)
        self.final = nn.Conv2d(base_features, 1, 1)

    def forward(self, x):
        d1 = self.down1(x)
        p1 = self.pool1(d1)
        d2 = self.down2(p1)
        p2 = self.pool2(d2)
        b = self.bridge(p2)
        u2 = self.up2(b)
        c2 = self.dec2(torch.cat([u2, d2], dim=1))
        u1 = self.up1(c2)
        c1 = self.dec1(torch.cat([u1, d1], dim=1))
        out = self.final(c1)
        return torch.sigmoid(out)  # output probability map

# Dice loss / score
def dice_score_tensor(y_true, y_pred, eps=1e-6):
    y_true = (y_true > 0.5).float()
    y_pred = (y_pred > 0.5).float()
    intersection = (y_true * y_pred).sum(dim=(1,2,3))
    union = y_true.sum(dim=(1,2,3)) + y_pred.sum(dim=(1,2,3))
    dice = (2 * intersection + eps) / (union + eps)
    return dice.mean()

def dice_loss(y_pred, y_true, eps=1e-6):
    y_true_f = (y_true > 0.5).float()
    intersection = (y_pred * y_true_f).sum(dim=(1,2,3))
    union = y_pred.sum(dim=(1,2,3)) + y_true_f.sum(dim=(1,2,3))
    loss = 1 - ((2 * intersection + eps) / (union + eps))
    return loss.mean()

# Wrapper dataset that applies the same resize / normalization as before
class FoldDataset(torch.utils.data.Dataset):
    def __init__(self, base_samples, resize_to=(256,256)):
        self.samples = base_samples
        self.resize_to = resize_to

    def __len__(self):
        return len(self.samples)

    def _load_image(self, path):
        img = cv2.imread(str(path), cv2.IMREAD_ANYDEPTH)
        if img is None:
            img = cv2.imread(str(path), cv2.IMREAD_COLOR)
            if img is None:
                raise RuntimeError(f"Failed to read image {path}")
            img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        # z-score + clamp as in preprocessing
        img = img.astype(np.float32)
        mean = img.mean()
        std = img.std() if img.std()>0 else 1.0
        img_norm = (img - mean)/std
        img_clipped = np.clip(img_norm, -3, 3)
        img_scaled = (img_clipped + 3) / 6
        resized = cv2.resize(img_scaled, (self.resize_to[1], self.resize_to[0]), interpolation=cv2.INTER_LINEAR)
        return np.expand_dims(resized, axis=0).astype(np.float32)  # CHW

    def _load_label(self, path):
        lbl = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if lbl is None:
            raise RuntimeError(f"Failed to read label {path}")
        lbl = (lbl > 0).astype(np.uint8)
        resized = cv2.resize(lbl, (self.resize_to[1], self.resize_to[0]), interpolation=cv2.INTER_NEAREST)
        return np.expand_dims(resized, axis=0).astype(np.float32)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        img = self._load_image(sample["img"])
        label = self._load_label(sample["label"])
        img_tensor = torch.from_numpy(img)
        label_tensor = torch.from_numpy(label)
        return img_tensor, label_tensor, sample["is_control"]

# Prepare for 5-fold CV: stratify on is_control to keep ratio
is_control_flags = np.array([int(s["is_control"]) for s in samples])
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

fold_results = []
for fold_idx, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(samples)), is_control_flags), start=1):
    print(f"\n=== Fold {fold_idx} ===")
    train_subset = [samples[i] for i in train_idx]
    val_subset = [samples[i] for i in val_idx]

    train_ds = FoldDataset(train_subset, resize_to=(256,256))
    val_ds = FoldDataset(val_subset, resize_to=(256,256))

    train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=0, pin_memory=torch.cuda.is_available())
    val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())

    model = UNetSimple(in_channels=1, base_features=32).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    best_val_dice = 0.0

    # Simple epoch loop (you can replace with early stopping, scheduler, etc.)
    for epoch in range(1, 21):  # 20 epochs per fold as example
        model.train()
        train_loss = 0.0
        for imgs, labels, _ in train_loader:
            imgs = imgs.to(device)
            labels = labels.to(device)
            preds = model(imgs)
            loss = dice_loss(preds, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * imgs.size(0)
        train_loss /= len(train_loader.dataset)

        # Validation
        model.eval()
        with torch.no_grad():
            val_losses = []
            val_dices = []
            for imgs, labels, _ in val_loader:
                imgs = imgs.to(device)
                labels = labels.to(device)
                preds = model(imgs)
                loss = dice_loss(preds, labels)
                dice = dice_score_tensor(preds, labels)
                val_losses.append(loss.item() * imgs.size(0))
                val_dices.append(dice.item() * imgs.size(0))
            val_loss = sum(val_losses)/len(val_loader.dataset)
            val_dice = sum(val_dices)/len(val_loader.dataset)

        print(f"Epoch {epoch}: train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_dice={val_dice:.4f}")

        if val_dice > best_val_dice:
            best_val_dice = val_dice
            # You could save best model per fold here
    fold_results.append(best_val_dice)
    print(f"Best validation Dice for fold {fold_idx}: {best_val_dice:.4f}")

# Summary across folds
mean_dice = np.mean(fold_results)
std_dice = np.std(fold_results)
print(f"\n5-Fold CV Dice: mean={mean_dice:.4f} std={std_dice:.4f}")